In [30]:
import mediapipe as mp
import cv2
import numpy as np

In [31]:
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Make Basic Detection

In [32]:
# Getting video feed
cap = cv2.VideoCapture(0)
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False 

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True 
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass

        # Rendering
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2),
                                  )

        cv2.imshow('Raw Webcam Feed', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

[x: 0.5913449
y: 0.62731296
z: -2.1631644
visibility: 0.99992883
, x: 0.6197529
y: 0.5309279
z: -2.0940998
visibility: 0.9998925
, x: 0.6450796
y: 0.5277126
z: -2.0953743
visibility: 0.99988866
, x: 0.666575
y: 0.5260222
z: -2.0953863
visibility: 0.9998623
, x: 0.5340165
y: 0.5272602
z: -2.1034443
visibility: 0.99994683
, x: 0.5095992
y: 0.5239003
z: -2.104438
visibility: 0.9999615
, x: 0.48555812
y: 0.52219987
z: -2.105393
visibility: 0.99995005
, x: 0.68392015
y: 0.5389899
z: -1.5424508
visibility: 0.99995303
, x: 0.43716675
y: 0.5555662
z: -1.5854849
visibility: 0.9999486
, x: 0.6329088
y: 0.70654655
z: -1.9376974
visibility: 0.9997501
, x: 0.5287548
y: 0.71714634
z: -1.9566336
visibility: 0.99977726
, x: 0.83561456
y: 0.8576636
z: -1.14999
visibility: 0.9961325
, x: 0.2894669
y: 0.91675407
z: -1.1852208
visibility: 0.99452555
, x: 0.986107
y: 1.2229555
z: -1.5910014
visibility: 0.28413987
, x: 0.20330787
y: 1.2991322
z: -1.7516854
visibility: 0.47692186
, x: 0.9380428
y: 1.1431981


# 2. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

In [33]:
landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]

x: 0.83940566
y: 0.8557808
z: -0.6272163
visibility: 0.9975505

In [34]:
landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]

x: 1.0235554
y: 1.2841911
z: -0.44453725
visibility: 0.25154868

# 3. Calculate Angles

In [35]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

In [36]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]


In [37]:
shoulder, elbow, wrist

([0.8394056558609009, 0.8557807803153992],
 [1.0235553979873657, 1.2841911315917969],
 [1.021963119506836, 1.6748019456863403])

In [38]:
calculate_angle(shoulder, elbow, wrist)

156.50626696365788

In [39]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # or any width you prefer
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)  # or any height you prefer
stage = ""
counter = 0
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, 
                  min_tracking_confidence=0.5,
                  model_complexity=2,
                  static_image_mode=False,
                  smooth_landmarks=False,
                          ) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
      
        # Make detection
        results = pose.process(image)
    
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        frame = cv2.flip(frame, 1)
        
        # Extract landmarks
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            for i, landmark in enumerate(landmarks):
                print(f"Landmark {i}: x={landmark.x}, y={landmark.y}, z={landmark.z}")

            # Get coordinates
            angles_to_calculate = {
                "right_elbow_right_shoulder_right_hip": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                ],
                "left_elbow_left_shoulder_left_hip": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                ],
                "right_knee_mid_hip_left_knee": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                ],
                "right_hip_right_knee_right_ankle": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
                ],
                "left_hip_left_knee_left_ankle": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                ],
                "right_wrist_right_elbow_right_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                ],
                "left_wrist_left_elbow_left_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                ],
            }

            # Calculate and visualize angles
            angles = []
            for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
                angle = calculate_angle(points[0], points[1], points[2])
                angles.append(angle)
                cv2.putText(image, f'{angle_name}: {int(angle)}', 
                            (50, 100 + i * 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

            if angles[3] > 160 and angles[4] > 160:
                stage = "up"
            if angles[3] < 90 and angles[4] < 90 and stage == "up":
                stage = "down"
                counter += 1
        else:
            print("No landmarks detected in this frame.")

            
        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (225,73), (245,117,16), -1)
        
        # Rep data
        cv2.putText(image, 'REPS', (15,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), 
                    (10,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Stage data
        cv2.putText(image, 'STAGE', (65,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, 
                    (60,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2) 
                                 )               
        
        # cv2.imshow('Mediapipe Feed', cv2.flip(image,1))
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

No landmarks detected in this frame.
Landmark 0: x=0.6219584941864014, y=0.7263088822364807, z=-0.8860396146774292
Landmark 1: x=0.6404075622558594, y=0.6446857452392578, z=-0.8572077751159668
Landmark 2: x=0.6562619209289551, y=0.6374392509460449, z=-0.8576940298080444
Landmark 3: x=0.6670683026313782, y=0.6316108107566833, z=-0.8575941920280457
Landmark 4: x=0.5816313624382019, y=0.64702308177948, z=-0.8611610531806946
Landmark 5: x=0.5639857053756714, y=0.6414614319801331, z=-0.8616964221000671
Landmark 6: x=0.5488656759262085, y=0.6366269588470459, z=-0.8620666265487671
Landmark 7: x=0.6698856949806213, y=0.6336228251457214, z=-0.6059233546257019
Landmark 8: x=0.5144992470741272, y=0.6453308463096619, z=-0.6236944794654846
Landmark 9: x=0.6453217267990112, y=0.7896140813827515, z=-0.7784148454666138
Landmark 10: x=0.579303503036499, y=0.8042646050453186, z=-0.7870907187461853
Landmark 11: x=0.767194926738739, y=0.8998262882232666, z=-0.46029895544052124
Landmark 12: x=0.40579289197

In [40]:
# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import nbimporter  # Import nbimporter

# from data import NeuralNetwork  
#  # Import your model class

# model = NeuralNetwork()  # Initialize the model
# model.load_state_dict(torch.load(r'D:\PROGRAMMING\BE proj24\data.pth'))  # Load the state dict
# model.eval() 

# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Get video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # or any width you prefer
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720) 
# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Prepare input data for your model
#             input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0)

#             # Predict next movement
#             with torch.no_grad():
#                 predicted_angles = model(input_tensor)
#                 predicted_angles = predicted_angles.squeeze().tolist()

#             # Display predicted movement
#             for i, angle in enumerate(predicted_angles):
#                 cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
#                             (50, 300 + i * 30),  # Adjust y-position to fit all angles
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA) 


#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                    mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()


In [41]:
# import sys
# sys.path.append(r'D:\PROGRAMMING\New folder\Rehabilitation-System\data.ipynb')
# import nbimporter   # Import nbimporter
# from data import NeuralNetwork

In [42]:
# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import torch.nn as nn
# from my_main import LSTMModel

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load the model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Instantiate the model
# lstm_model = LSTMModel().to(device)

# # Load the state dictionary
# lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# # Set the model to evaluation mode
# lstm_model.eval()
# # Set up MediaPipe drawing and pose
# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Open video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# stage = ""
# counter = 0

# frame_data=[]
# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make pose detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks and calculate angles
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Define angle calculations (you can add other angles here)
#             angles_to_calculate = {
#                 "right_elbow_right_shoulder_right_hip": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                 ],
#                 "left_elbow_left_shoulder_left_hip": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                 ],
#                 "right_knee_mid_hip_left_knee": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
#                      (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                 ],
#                 "right_hip_right_knee_right_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
#                 ],
#                 "left_hip_left_knee_left_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
#                 ],
#                 "right_wrist_right_elbow_right_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                 ],
#                 "left_wrist_left_elbow_left_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                 ],
#             }

#             # Calculate angles and store them in the list for model input
#             angles = []
#             for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
#                 angle = calculate_angle(points[0], points[1], points[2])
#                 angles.append(angle)
#                 cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

#             # Store angles for multiple frames
#             frame_data.append(angles)

#             # Ensure there are 50 frames of data
#             if len(frame_data) > 50:
#                 frame_data.pop(0)

#             # Only make prediction if we have 50 frames
#             if len(frame_data) == 50:
#                 input_tensor = torch.tensor(frame_data, dtype=torch.float32).unsqueeze(0).to(device)

#                 # Predict next movement
#                 with torch.no_grad():
#                     predicted_angles = lstm_model(input_tensor).squeeze().tolist()

#                 # Display predicted next movement angles
#                 for i, angle in enumerate(predicted_angles):
#                     cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
#                                 (50, 300 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

#             # Rep counting logic
#             if angles[3] > 160 and angles[4] > 160:
#                 stage = "up"
#             if angles[3] < 90 and angles[4] < 90 and stage == "up":
#                 stage = "down"
#                 counter += 1

#         # Render rep counter and stage data
#         cv2.rectangle(image, (0, 0), (225, 73), (245, 117, 16), -1)
#         cv2.putText(image, 'REPS', (15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#         cv2.putText(image, str(counter), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)
#         cv2.putText(image, 'STAGE', (65, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#         cv2.putText(image, stage, (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)

#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                   mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         # Display image
#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()

In [43]:


# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import torch.nn as nn
# from my_main import LSTMModel

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load the model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Instantiate the model
# lstm_model = LSTMModel().to(device)

# # Load the state dictionary
# lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# # Set the model to evaluation mode
# lstm_model.eval()
# # Set up MediaPipe drawing and pose
# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Open video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# stage = ""
# counter = 0

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make pose detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks and calculate angles
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Define angle calculations (you can add other angles here)
#             angles_to_calculate = {
#                 "right_elbow_right_shoulder_right_hip": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                 ],
#                 "left_elbow_left_shoulder_left_hip": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                 ],
#                 "right_knee_mid_hip_left_knee": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
#                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                 ],
#                 "right_hip_right_knee_right_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
#                 ],
#                 "left_hip_left_knee_left_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
#                 ],
#                 "right_wrist_right_elbow_right_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                 ],
#                 "left_wrist_left_elbow_left_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                 ],
#             }

#             # Calculate current angles and store them in the list for model input
#             angles = []
#             for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
#                 angle = calculate_angle(points[0], points[1], points[2])
#                 angles.append(angle)
#                 cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

#             input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0).to(device)
#             input_tensor = input_tensor.repeat(64, 50, 1).to(device)
#             # Predict next movement
#             with torch.no_grad():
#                 predicted_angles = lstm_model(input_tensor).squeeze().tolist()

#             # Display predicted next movement angles
#             for i, angle in enumerate(predicted_angles):
#                 # Ensure angle is numeric
#                 if isinstance(angle, list):
#                     # If the angle is a list, flatten it and display each item
#                     for j, sub_angle in enumerate(angle):
#                         if isinstance(sub_angle, (int, float)):  # Ensure it's a number
#                             y_pos = 300 + (i * 30) + (j * 20)  # Adjusted y position for sub-angles
#                             cv2.putText(image, f'Predicted Angle {i+1}-{j+1}: {int(sub_angle)}', 
#                                         (300, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
#                 elif isinstance(angle, (int, float)):  # Ensure the angle is numeric
#                     y_pos = 300 + i * 30  # Adjusted y position for main angles
#                     cv2.putText(image, f'Predicted Angle {i+1}: {int(angle)}', 
#                                 (300, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                   mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         # Display image
#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()

In [44]:


import mediapipe as mp
import cv2
import numpy as np
import torch
import torch.nn as nn
from my_main import LSTMModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the model
lstm_model = LSTMModel().to(device)

# Load the state dictionary
lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# Set the model to evaluation mode
lstm_model.eval()
# Set up MediaPipe drawing and pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Open video feed
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
stage = ""
counter = 0

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make pose detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks and calculate angles
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Define angle calculations (you can add other angles here)
            angles_to_calculate = {
                "right_elbow_right_shoulder_right_hip": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                ],
                "left_elbow_left_shoulder_left_hip": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                ],
                "right_knee_mid_hip_left_knee": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                ],
                "right_hip_right_knee_right_ankle": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
                ],
                "left_hip_left_knee_left_ankle": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                ],
                "right_wrist_right_elbow_right_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                ],
                "left_wrist_left_elbow_left_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                ],
            }

            # Calculate angles and store them in the list for model input
            angles = []
            for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
                angle = calculate_angle(points[0], points[1], points[2])
                angles.append(angle)
                cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

            input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0).to(device)
            input_tensor = input_tensor.repeat(64, 50, 1).to(device)
            # Predict next movement
            with torch.no_grad():
                predicted_angles = lstm_model(input_tensor).squeeze().tolist()

            # Display predicted next movement angles
            for i, angle in enumerate(predicted_angles):
                if isinstance(angle, list):
                    for j, sub_angle in enumerate(angle):
                        if isinstance(sub_angle, (int, float)):
                            # Change color for predicted angles to blue
                            cv2.putText(image, f'Next Angle {i+1}-{j+1}: {int(sub_angle)}', 
                                        (50, 300 + (i * 30) + (j * 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2, cv2.LINE_AA)
                elif isinstance(angle, (int, float)):
                    # Change color for predicted angles to blue
                    cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
                                (50, 300 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2, cv2.LINE_AA)


            # Rep counting logic
            if angles[3] > 160 and angles[4] > 160:
                stage = "up"
            if angles[3] < 90 and angles[4] < 90 and stage == "up":
                stage = "down"
                counter += 1

        # Render rep counter and stage data
        cv2.rectangle(image, (0, 0), (225, 73), (245, 117, 16), -1)
        cv2.putText(image, 'REPS', (15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(image, 'STAGE', (65, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)

        # Render pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

        # Display image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [45]:
# import os
# print(os.path.isfile(r'D:\PROGRAMMING\New folder\Rehabilitation-System\data.ipynb'))

# Holistic

In [46]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [47]:
import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

cap = cv2.VideoCapture(0)

# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, 
                          min_tracking_confidence=0.5,
                          static_image_mode=False,
                          smooth_landmarks=True,
                          model_complexity=2
                            ) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )

        # 3. Left Hand
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )
                        
        cv2.imshow('Holistic Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [48]:
# SPINE ANGLE

In [49]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a = np.array(a)  # First point  
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates for back straightness check
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
            # Calculate angle for back straightness
            spine_angle = calculate_angle(shoulder, hip, knee)
            
            # Visualize spine angle
            cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
                        (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
            # Get foot positions for feet check
            left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
            right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
            # Check if feet are lifting off the ground
            if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
                cv2.putText(image, "Feet not planted!", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(image, "Feet planted", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
        except:
            pass
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
        cv2.imshow('Squat Form Detection', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


# Holistic

In [50]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [51]:
import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

cap = cv2.VideoCapture(0)

# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, 
                          min_tracking_confidence=0.5,
                          static_image_mode=False,
                          smooth_landmarks=True,
                          model_complexity=2
                            ) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )

        # 3. Left Hand
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )
                        
        cv2.imshow('Holistic Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [52]:
# SPINE ANGLE

In [53]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a = np.array(a)  # First point  
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates for back straightness check
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
            # Calculate angle for back straightness
            spine_angle = calculate_angle(shoulder, hip, knee)
            
            # Visualize spine angle
            cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
                        (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
            # Get foot positions for feet check
            left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
            right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
            # Check if feet are lifting off the ground
            if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
                cv2.putText(image, "Feet not planted!", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(image, "Feet planted", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
        except:
            pass
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
        cv2.imshow('Squat Form Detection', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
